<a href="https://colab.research.google.com/github/hbhoomikacs24-stack/lab-programs-for-AI/blob/main/week7_1BM24CS107_resolution_fol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re

def parse_literal(lit):
    lit = lit.strip()
    neg = lit.startswith('~')
    if neg:
        lit = lit[1:].strip()
    match = re.match(r"([A-Za-z0-9_]+)\(([^)]*)\)", lit)
    if not match:
        raise ValueError(f"Invalid literal format: {lit}")
    pred = match.group(1)
    args = tuple(arg.strip() for arg in match.group(2).split(','))
    return (neg, pred, args)

def format_literal(lit_tuple):
    neg, pred, args = lit_tuple
    prefix = "~" if neg else ""
    return f"{prefix}{pred}({', '.join(args)})"

def format_clause(clause):
    if not clause:
        return "EMPTY CLAUSE (Contradiction Found!)"
    return " | ".join(sorted(format_literal(l) for l in clause))

def is_variable(term):
    return isinstance(term, str) and term.islower()

def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    elif isinstance(x, str) and x in theta:
        return unify(var, theta[x], theta)
    elif occur_check(var, x, theta):
        return None
    else:
        theta_copy = theta.copy()
        theta_copy[var] = x
        return theta_copy

def occur_check(var, x, theta):
    if var == x:
        return True
    elif is_variable(x) and x in theta:
        return occur_check(var, theta[x], theta)
    return False

def unify(x, y, theta):
    if theta is None:
        return None
    elif x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for xi, yi in zip(x, y):
            theta = unify(xi, yi, theta)
        return theta
    else:
        return None

def apply_substitution(literal, theta):
    neg, pred, args = literal
    new_args = []
    for arg in args:
        while is_variable(arg) and arg in theta:
            arg = theta[arg]
        new_args.append(arg)
    return (neg, pred, tuple(new_args))

def standardize_variables(clause, counter):
    mapping = {}
    new_clause = []
    for neg, pred, args in clause:
        new_args = []
        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = f"v{counter}"
                    counter += 1
                new_args.append(mapping[arg])
            else:
                new_args.append(arg)
        new_clause.append((neg, pred, tuple(new_args)))
    return frozenset(new_clause), counter

def canonical_form(clause):
    """Normalizes variable names starting from v1 to detect duplicate clauses."""
    sorted_lits = sorted(clause, key=lambda l: (l[1], l[0], l[2]))
    mapping = {}
    counter = 1
    canon_lits = []
    for neg, pred, args in sorted_lits:
        new_args = []
        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = f"v{counter}"
                    counter += 1
                new_args.append(mapping[arg])
            else:
                new_args.append(arg)
        canon_lits.append((neg, pred, tuple(new_args)))
    return frozenset(canon_lits)

def resolve(ci, cj):
    resolvents = []
    for li in ci:
        for lj in cj:
            if li[1] == lj[1] and li[0] != lj[0]:
                theta = unify(li[2], lj[2], {})
                if theta is not None:
                    new_ci = [apply_substitution(l, theta) for l in ci if l != li]
                    new_cj = [apply_substitution(l, theta) for l in cj if l != lj]
                    resolvent = frozenset(new_ci + new_cj)
                    resolvents.append((resolvent, theta))
    return resolvents

def fol_resolution(kb_clauses, negated_goal_clauses):
    clauses = []
    seen = set()
    resolved_pairs = set()
    var_counter = 1

    for c in kb_clauses + negated_goal_clauses:
        std_c, var_counter = standardize_variables(c, var_counter)
        canon = canonical_form(std_c)
        if canon not in seen:
            seen.add(canon)
            clauses.append(std_c)

    print("\n--- Initial Clauses (KB + Negated Goal) ---")
    for idx, c in enumerate(clauses, 1):
        print(f"C{idx}: {format_clause(c)}")
    print("\n--- Resolution Steps ---")

    step = len(clauses) + 1
    while True:
        new_clauses = []
        n = len(clauses)
        for i in range(n):
            for j in range(i + 1, n):
                if (i, j) in resolved_pairs:
                    continue
                resolved_pairs.add((i, j))

                resolvents = resolve(clauses[i], clauses[j])
                for res, theta in resolvents:
                    std_res, var_counter = standardize_variables(res, var_counter)
                    canon = canonical_form(std_res)
                    if canon not in seen:
                        sub_str = f" with {theta}" if theta else ""
                        print(f"C{step}: {format_clause(res)}  [from C{i+1} & C{j+1}{sub_str}]")
                        step += 1
                        if len(res) == 0:
                            return True
                        seen.add(canon)
                        new_clauses.append(std_res)

        if not new_clauses:
            return False
        clauses.extend(new_clauses)

def parse_clause_input(raw_str):
    literals = raw_str.split('|')
    return [parse_literal(l) for l in literals if l.strip()]

if __name__ == "__main__":
    print("=== General First-Order Logic (FOL) Resolution Prover ===")
    print("Rules for Input:")
    print("  1. Use '~' for NOT, '|' for OR.")
    print("  2. Variables MUST start with lowercase (e.g., x, y, z).")
    print("  3. Constants & Predicates MUST start with UPPERCASE (e.g., Alice, Likes(x, y)).")
    print("  4. Enter premises in CNF form.\n")

    num_premises = int(input("Enter the number of CNF premises in KB: "))
    kb = []
    for i in range(num_premises):
        raw = input(f"Premise {i+1}: ")
        kb.append(parse_clause_input(raw))

    print("\nEnter the NEGATED Goal clause(s) (If goal is D(Alice), enter ~D(Alice)):")
    num_goal = int(input("Number of negated goal clauses (usually 1): "))
    negated_goal = []
    for i in range(num_goal):
        raw = input(f"Negated Goal {i+1}: ")
        negated_goal.append(parse_clause_input(raw))

    result = fol_resolution(kb, negated_goal)

    print("\n==========================================")
    if result:
        print("RESULT: PROVED! (Contradiction reached; Goal is logically entailed)")
    else:
        print("RESULT: CANNOT BE PROVED! (No contradiction found; Goal is not entailed)")
    print("==========================================")


=== General First-Order Logic (FOL) Resolution Prover ===
Rules for Input:
  1. Use '~' for NOT, '|' for OR.
  2. Variables MUST start with lowercase (e.g., x, y, z).
  3. Constants & Predicates MUST start with UPPERCASE (e.g., Alice, Likes(x, y)).
  4. Enter premises in CNF form.

Enter the number of CNF premises in KB: 9
Premise 1: ~Food(x) | Likes(John, x)
Premise 2: Food(Apple)
Premise 3: Food(Vegetables)
Premise 4: ~Eats(y, z) | Killed(y) | Food(z)
Premise 5: Eats(Anil, Peanuts)
Premise 6: Alive(Anil)
Premise 7: ~Eats(Anil, w) | Eats(Harry, w)
Premise 8: ~Alive(u) | ~Killed(u)
Premise 9: Killed(v) | Alive(v)

Enter the NEGATED Goal clause(s) (If goal is D(Alice), enter ~D(Alice)):
Number of negated goal clauses (usually 1): 1
Negated Goal 1: ~Likes(John, Peanuts)

--- Initial Clauses (KB + Negated Goal) ---
C1: Likes(John, v1) | ~Food(v1)
C2: Food(Apple)
C3: Food(Vegetables)
C4: Food(v3) | Killed(v2) | ~Eats(v2, v3)
C5: Eats(Anil, Peanuts)
C6: Alive(Anil)
C7: Eats(Harry, v4) | ~Ea